In [ ]:
# ============================================================
# SimpleGenie の実装
# ============================================================

class SimpleGenie(nn.Module):
    """SimpleGenie: 全コンポーネントを統合したインタラクティブ世界生成モデル
    
    訓練済みの各コンポーネントを組み合わせて、
    初期フレーム + 行動列 → 動画生成 を実現する。
    """
    
    def __init__(self, frame_encoder, action_encoder, dynamics_model, frame_decoder):
        super().__init__()
        self.frame_encoder = frame_encoder
        self.action_encoder = action_encoder
        self.dynamics_model = dynamics_model
        self.frame_decoder = frame_decoder
    
    @torch.no_grad()
    def generate_video(self, initial_frame, action_sequence):
        """初期フレームと行動列から動画を生成
        
        Args:
            initial_frame: (1, C, H, W) 初期フレーム
            action_sequence: list of int, 行動インデックスの列
        Returns:
            frames: list of (C, H, W) tensors, 生成されたフレーム列
        """
        self.eval()
        
        frames = [initial_frame.squeeze(0).cpu()]
        
        # 初期フレームをエンコード
        z_current = self.frame_encoder(initial_frame)
        
        for action_idx in action_sequence:
            # 行動テンソルを作成
            action_tensor = torch.tensor([action_idx],
                                          device=initial_frame.device,
                                          dtype=torch.long)
            
            # DynamicsModelで次の潜在状態を予測
            z_next = self.dynamics_model(z_current, action_tensor)
            
            # 潜在状態からフレームをデコード
            # デコーダは (z_frame, z_action) を受け取るので、
            # z_action にはダミーを入れる（もしくはz_nextのみで予測）
            # ここでは z_current と action_embedding の和を z_action として使用
            action_emb = self.dynamics_model.action_embedding(action_tensor)
            x_next = self.frame_decoder(z_next, action_emb)
            
            frames.append(x_next.squeeze(0).cpu())
            z_current = z_next
        
        return frames
    
    @torch.no_grad()
    def discover_action(self, x_t, x_t1):
        """2フレームから行動を発見"""
        self.eval()
        _, _, action_idx = self.action_encoder(x_t, x_t1)
        return action_idx

# --- SimpleGenieの構築 ---
genie = SimpleGenie(
    frame_encoder=discovery_model.frame_encoder,
    action_encoder=discovery_model.action_encoder,
    dynamics_model=dynamics_model,
    frame_decoder=discovery_model.frame_decoder,
).to(device)

print("SimpleGenie 構築完了")
print(f"  コンポーネント:")
print(f"    - FrameEncoder")
print(f"    - LatentActionEncoder (codebook_size=8)")
print(f"    - DynamicsModel (Transformer 2層)")
print(f"    - FrameDecoder")
total = sum(p.numel() for p in genie.parameters())
print(f"  総パラメータ数: {total:,}")

<a id="section7"></a>
## 7. SimpleGenie — 統合モデル

### 📊 全コンポーネントの統合

これまでに実装した3つのコンポーネントを統合して、**SimpleGenie**を構築します。

```
【訓練済みコンポーネント】

1. FrameEncoder: フレーム → 潜在表現
2. LatentActionEncoder: フレーム対 → 離散行動トークン
3. DynamicsModel: 潜在状態 + 行動 → 次の潜在状態
4. FrameDecoder: 潜在表現 → フレーム

【推論時のフロー】

初期フレーム x_0  →  FrameEncoder  →  z_0
                                         ↓
         ユーザー行動 a_0  →  DynamicsModel  →  z_1  →  FrameDecoder  →  x_1
                                                          ↓
                  ユーザー行動 a_1  →  DynamicsModel  →  z_2  →  FrameDecoder  →  x_2
                                                                   ↓
                                                                  ...
```

In [ ]:
# ============================================================
# DynamicsModel 損失推移
# ============================================================

plt.figure(figsize=(10, 5))
plt.plot(dynamics_losses, 'b-', linewidth=2, marker='o', markersize=4)
plt.xlabel('エポック', fontsize=12)
plt.ylabel('損失 (MSE in latent space)', fontsize=12)
plt.title('DynamicsModel 訓練損失\n（潜在空間での次フレーム予測精度）',
          fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# DynamicsModel の訓練
# 学習済みのFrameEncoderとLatentActionEncoderを使用
# ============================================================

# 学習済みモデルからフレームエンコーダとデコーダを取得
frame_encoder = discovery_model.frame_encoder
frame_decoder = discovery_model.frame_decoder

# 新しいDynamicsModelを初期化
torch.manual_seed(42)
dynamics_model = DynamicsModel(
    hidden_dim=64, codebook_size=8, num_heads=4, num_layers=2
).to(device)
dynamics_optimizer = torch.optim.Adam(dynamics_model.parameters(), lr=3e-4)

print("=" * 60)
print("DynamicsModel の訓練")
print("=" * 60)

# 訓練データを準備: フレーム潜在 + 発見行動 → 次フレーム潜在
frame_encoder.eval()
discovery_model.eval()

dynamics_losses = []

for epoch in range(20):
    dynamics_model.train()
    epoch_loss = 0
    n_batches = 0
    
    for x_t, x_t1, _ in train_loader:
        x_t = x_t.to(device)
        x_t1 = x_t1.to(device)
        
        with torch.no_grad():
            # フレームの潜在表現
            z_t = frame_encoder(x_t)          # (B, 64)
            z_t1 = frame_encoder(x_t1)        # (B, 64) ← ターゲット
            
            # 発見された行動
            _, _, action_indices = discovery_model.action_encoder(x_t, x_t1)
        
        # DynamicsModelで次フレーム潜在を予測
        z_pred = dynamics_model(z_t, action_indices)
        
        # 潜在空間での予測損失
        loss = F.mse_loss(z_pred, z_t1)
        
        dynamics_optimizer.zero_grad()
        loss.backward()
        dynamics_optimizer.step()
        
        epoch_loss += loss.item()
        n_batches += 1
    
    avg_loss = epoch_loss / n_batches
    dynamics_losses.append(avg_loss)
    
    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1:3d}/20 | Dynamics Loss: {avg_loss:.6f}")

print("DynamicsModel 訓練完了")

In [ ]:
# ============================================================
# DynamicsModel の実装
# ============================================================

class DynamicsModel(nn.Module):
    """Transformerベースの動態モデル
    
    現在のフレーム潜在表現と行動トークンから、次のフレーム潜在表現を予測する。
    
    Args:
        hidden_dim (int): 潜在表現の次元
        codebook_size (int): 行動コードブックのサイズ
        num_heads (int): Transformerのアテンションヘッド数
        num_layers (int): Transformerのレイヤー数
    """
    
    def __init__(self, hidden_dim=64, codebook_size=8, num_heads=4, num_layers=2):
        super().__init__()
        self.hidden_dim = hidden_dim
        
        # 行動トークンの埋め込み
        self.action_embedding = nn.Embedding(codebook_size, hidden_dim)
        
        # 入力の射影: フレーム潜在 + 行動埋め込み → Transformer入力
        self.input_proj = nn.Linear(hidden_dim * 2, hidden_dim)
        
        # Transformerエンコーダ（2層）
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=0.1,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # 出力射影: → 次フレームの潜在表現
        self.output_proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
    
    def forward(self, z_frame, action_indices):
        """次フレームの潜在表現を予測
        
        Args:
            z_frame: (B, hidden_dim) 現在のフレーム潜在表現
            action_indices: (B,) 行動トークンのインデックス
        Returns:
            z_next: (B, hidden_dim) 予測された次フレームの潜在表現
        """
        # 行動埋め込みを取得
        z_action = self.action_embedding(action_indices)  # (B, hidden_dim)
        
        # フレーム潜在と行動を結合
        z_combined = torch.cat([z_frame, z_action], dim=1)  # (B, hidden_dim*2)
        z_input = self.input_proj(z_combined)                 # (B, hidden_dim)
        
        # Transformer（シーケンス長1として処理）
        z_input = z_input.unsqueeze(1)                        # (B, 1, hidden_dim)
        z_out = self.transformer(z_input)                     # (B, 1, hidden_dim)
        z_out = z_out.squeeze(1)                              # (B, hidden_dim)
        
        # 残差接続 + 出力射影
        z_next = z_frame + self.output_proj(z_out)            # (B, hidden_dim)
        
        return z_next

# --- テスト ---
dynamics = DynamicsModel(hidden_dim=64, codebook_size=8, num_heads=4, num_layers=2)
z_frame_test = torch.randn(4, 64)
action_test = torch.tensor([0, 1, 2, 3])
z_next_test = dynamics(z_frame_test, action_test)

print("DynamicsModel テスト:")
print(f"  フレーム潜在入力: {z_frame_test.shape}")
print(f"  行動入力: {action_test.tolist()}")
print(f"  次フレーム潜在出力: {z_next_test.shape}")
total_params = sum(p.numel() for p in dynamics.parameters())
print(f"  パラメータ数: {total_params:,}")

<a id="section6"></a>
## 6. DynamicsModel — 次フレーム予測

### 📊 Transformerベースの動態モデル

Genieの論文では、Dynamics Modelに**ST-Transformer（Spatio-Temporal Transformer）**を
使用しています。ここでは簡略化した**Transformer**で次フレームの潜在表現を予測します。

動態モデルの役割:
- 入力: 現在のフレーム潜在表現 + 潜在行動トークン
- 出力: 次のフレーム潜在表現
- 推論時: ユーザーが行動を選択 → 次フレームを自己回帰的に生成

In [ ]:
# ============================================================
# t-SNE プロット: 発見された行動 vs 真の行動
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- (a) 真の行動で色分け ---
action_colors = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12']
action_labels = GridWorldDataset.ACTION_NAMES

for act_id in range(4):
    mask = all_true == act_id
    axes[0].scatter(z_2d[mask, 0], z_2d[mask, 1],
                    c=action_colors[act_id], s=15, alpha=0.6,
                    label=f'{action_labels[act_id]} ({act_id})')
axes[0].set_title('真の行動ラベルで色分け', fontsize=13, fontweight='bold')
axes[0].set_xlabel('t-SNE 次元1', fontsize=11)
axes[0].set_ylabel('t-SNE 次元2', fontsize=11)
axes[0].legend(fontsize=10, title='真の行動')
axes[0].grid(True, alpha=0.3)

# --- (b) 発見された行動で色分け ---
cmap = plt.cm.get_cmap('tab10')
unique_discovered = np.unique(all_discovered)
for act_id in unique_discovered:
    mask = all_discovered == act_id
    axes[1].scatter(z_2d[mask, 0], z_2d[mask, 1],
                    c=[cmap(act_id / max(unique_discovered.max(), 1))],
                    s=15, alpha=0.6, label=f'コード {act_id}')
axes[1].set_title('発見された潜在行動で色分け', fontsize=13, fontweight='bold')
axes[1].set_xlabel('t-SNE 次元1', fontsize=11)
axes[1].set_ylabel('t-SNE 次元2', fontsize=11)
axes[1].legend(fontsize=9, title='発見行動', ncol=2)
axes[1].grid(True, alpha=0.3)

plt.suptitle('潜在行動空間の t-SNE 可視化\n'
             '（左: 真の行動 / 右: ラベルなしで発見された行動）',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("左右のプロットを比較してください。")
print("発見された離散行動が、真の行動のクラスタに対応していれば成功です。")

In [ ]:
# ============================================================
# 潜在行動のt-SNE可視化（発見された行動 vs 真の行動）
# ============================================================

discovery_model.eval()

# 全データに対して潜在行動ベクトルを取得
all_z_actions = []
all_discovered = []
all_true = []

with torch.no_grad():
    for i in range(len(dataset)):
        ft, ft1, true_act = dataset[i]
        ft_d = ft.unsqueeze(0).to(device)
        ft1_d = ft1.unsqueeze(0).to(device)
        
        # 行動エンコーダの中間表現を取得
        x_pair = torch.cat([ft_d, ft1_d], dim=1)
        z_e = discovery_model.action_encoder.encoder(x_pair)
        z_q, _, act_idx = discovery_model.action_encoder.vq(z_e)
        
        all_z_actions.append(z_e.cpu().numpy())
        all_discovered.append(act_idx.item())
        all_true.append(true_act)

all_z_actions = np.concatenate(all_z_actions, axis=0)
all_discovered = np.array(all_discovered)
all_true = np.array(all_true)

print(f"潜在行動ベクトル: {all_z_actions.shape}")
print(f"発見された行動の分布: {dict(zip(*np.unique(all_discovered, return_counts=True)))}")

# t-SNE
print("t-SNEを計算中...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
z_2d = tsne.fit_transform(all_z_actions)
print("t-SNE完了")

In [ ]:
# ============================================================
# 再構成結果の確認: x_t + 発見行動 → 予測 x_{t+1} vs 真の x_{t+1}
# ============================================================

discovery_model.eval()

fig, axes = plt.subplots(4, 6, figsize=(15, 10))
col_titles = ['$x_t$（入力）', '$x_{t+1}$（真）', '$\\hat{x}_{t+1}$（予測）',
              '$x_t$（入力）', '$x_{t+1}$（真）', '$\\hat{x}_{t+1}$（予測）']

sample_indices = [0, 500, 1000, 1500]

for row, idx in enumerate(sample_indices):
    ft, ft1, true_act = dataset[idx]
    ft_d = ft.unsqueeze(0).to(device)
    ft1_d = ft1.unsqueeze(0).to(device)
    
    with torch.no_grad():
        x_pred, _, discovered_act = discovery_model(ft_d, ft1_d)
    
    for col_group in range(2):
        base_col = col_group * 3
        if col_group == 0:
            # 左3列: このサンプル
            axes[row, base_col].imshow(ft.permute(1, 2, 0).numpy())
            axes[row, base_col + 1].imshow(ft1.permute(1, 2, 0).numpy())
            axes[row, base_col + 2].imshow(x_pred[0].cpu().permute(1, 2, 0).numpy())
            
            axes[row, base_col].set_ylabel(
                f'真: {GridWorldDataset.ACTION_NAMES[true_act]}\n'
                f'発見: {discovered_act.item()}',
                fontsize=10)
        else:
            # 右3列: 別のサンプル
            idx2 = idx + 250
            if idx2 < len(dataset):
                ft2, ft12, true_act2 = dataset[idx2]
                ft2_d = ft2.unsqueeze(0).to(device)
                ft12_d = ft12.unsqueeze(0).to(device)
                with torch.no_grad():
                    x_pred2, _, disc_act2 = discovery_model(ft2_d, ft12_d)
                axes[row, base_col].imshow(ft2.permute(1, 2, 0).numpy())
                axes[row, base_col + 1].imshow(ft12.permute(1, 2, 0).numpy())
                axes[row, base_col + 2].imshow(x_pred2[0].cpu().permute(1, 2, 0).numpy())
    
    for col in range(6):
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(col_titles[col], fontsize=10)

plt.suptitle('行動発見モデルの再構成結果\n'
             '（行動ラベルなしで学習、次フレーム予測）',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 訓練損失の可視化
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(train_losses, 'b-', linewidth=2)
axes[0].set_xlabel('エポック', fontsize=12)
axes[0].set_ylabel('総損失', fontsize=12)
axes[0].set_title('総損失', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(recon_losses, 'g-', linewidth=2)
axes[1].set_xlabel('エポック', fontsize=12)
axes[1].set_ylabel('再構成損失', fontsize=12)
axes[1].set_title('再構成損失（次フレーム予測）', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

axes[2].plot(vq_losses, 'r-', linewidth=2)
axes[2].set_xlabel('エポック', fontsize=12)
axes[2].set_ylabel('VQ損失', fontsize=12)
axes[2].set_title('VQ損失（コードブック + コミットメント）', fontsize=13, fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.suptitle('行動発見モデルの訓練損失', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 行動発見モデルの訓練
# ============================================================

# ハイパーパラメータ
BATCH_SIZE = 64
NUM_EPOCHS = 20
LR = 3e-4
VQ_WEIGHT = 1.0  # VQ損失の重み

# モデル初期化
torch.manual_seed(42)
discovery_model = ActionDiscoveryModel(
    in_channels=3, hidden_dim=64, codebook_size=8
).to(device)
optimizer = torch.optim.Adam(discovery_model.parameters(), lr=LR)

# データローダー
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

print("=" * 60)
print("潜在行動の学習（行動ラベルなし）")
print("=" * 60)
print(f"  バッチサイズ: {BATCH_SIZE}")
print(f"  エポック数: {NUM_EPOCHS}")
print(f"  コードブックサイズ: 8")
print(f"  学習率: {LR}")
print("-" * 60)

# 訓練ループ
train_losses = []
recon_losses = []
vq_losses = []

for epoch in range(NUM_EPOCHS):
    discovery_model.train()
    epoch_loss = 0
    epoch_recon = 0
    epoch_vq = 0
    n_batches = 0
    
    for x_t, x_t1, _ in train_loader:  # ラベル(_)は使わない!
        x_t = x_t.to(device)
        x_t1 = x_t1.to(device)
        
        # 順伝播
        x_pred, vq_loss, action_indices = discovery_model(x_t, x_t1)
        
        # 再構成損失: 次フレームの予測精度
        recon_loss = F.mse_loss(x_pred, x_t1)
        
        # 全体損失
        loss = recon_loss + VQ_WEIGHT * vq_loss
        
        # 逆伝播
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(discovery_model.parameters(), 1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
        epoch_recon += recon_loss.item()
        epoch_vq += vq_loss.item()
        n_batches += 1
    
    avg_loss = epoch_loss / n_batches
    avg_recon = epoch_recon / n_batches
    avg_vq = epoch_vq / n_batches
    train_losses.append(avg_loss)
    recon_losses.append(avg_recon)
    vq_losses.append(avg_vq)
    
    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1:3d}/{NUM_EPOCHS} | "
              f"Loss: {avg_loss:.4f} (Recon: {avg_recon:.4f}, VQ: {avg_vq:.4f})")

print("-" * 60)
print("訓練完了")

In [ ]:
# ============================================================
# 行動発見のための学習モデル
# LatentActionEncoder + フレームエンコーダ/デコーダ
# ============================================================

class FrameEncoder(nn.Module):
    """フレームエンコーダ: 画像 → 潜在ベクトル"""
    
    def __init__(self, in_channels=3, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, stride=2, padding=1),  # 16→8
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),          # 8→4
            nn.ReLU(),
            nn.Conv2d(64, hidden_dim, 3, stride=2, padding=1),  # 4→2
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
        )
    
    def forward(self, x):
        return self.net(x)  # (B, hidden_dim)


class FrameDecoder(nn.Module):
    """フレームデコーダ: 潜在ベクトル → 画像"""
    
    def __init__(self, out_channels=3, hidden_dim=64):
        super().__init__()
        # 入力: フレーム潜在 + 行動潜在を結合
        self.fc = nn.Linear(hidden_dim * 2, hidden_dim * 4)
        self.net = nn.Sequential(
            # (B, hidden_dim, 2, 2)
            nn.ConvTranspose2d(hidden_dim, 64, 4, stride=2, padding=1),  # 2→4
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),         # 4→8
            nn.ReLU(),
            nn.ConvTranspose2d(32, out_channels, 4, stride=2, padding=1),# 8→16
            nn.Sigmoid(),  # [0, 1] に制限
        )
    
    def forward(self, z_frame, z_action):
        """
        Args:
            z_frame: (B, hidden_dim) フレームの潜在表現
            z_action: (B, hidden_dim) 行動の潜在表現
        Returns:
            (B, C, H, W) 予測された次フレーム
        """
        z = torch.cat([z_frame, z_action], dim=1)  # (B, hidden_dim*2)
        h = self.fc(z)                              # (B, hidden_dim*4)
        h = h.view(-1, 64, 2, 2)                    # 空間次元に変形
        return self.net(h)                           # (B, C, 16, 16)


class ActionDiscoveryModel(nn.Module):
    """行動発見モデル: フレーム対から行動を発見し、次フレームを予測
    
    LatentActionEncoder + FrameEncoder + FrameDecoder の統合
    """
    
    def __init__(self, in_channels=3, hidden_dim=64, codebook_size=8):
        super().__init__()
        self.frame_encoder = FrameEncoder(in_channels, hidden_dim)
        self.action_encoder = LatentActionEncoder(in_channels, hidden_dim, codebook_size)
        self.frame_decoder = FrameDecoder(in_channels, hidden_dim)
    
    def forward(self, x_t, x_t1):
        """順伝播
        
        Args:
            x_t:  (B, C, H, W) 現在のフレーム
            x_t1: (B, C, H, W) 次のフレーム（再構成ターゲット）
        Returns:
            x_pred: (B, C, H, W) 予測された次フレーム
            vq_loss: VQ損失
            action_indices: (B,) 発見された離散行動
        """
        # フレームtの潜在表現
        z_frame = self.frame_encoder(x_t)
        
        # フレーム対から行動を抽出（VQ量子化）
        z_action, vq_loss, action_indices = self.action_encoder(x_t, x_t1)
        
        # フレーム潜在 + 行動潜在 → 次フレームを予測
        x_pred = self.frame_decoder(z_frame, z_action)
        
        return x_pred, vq_loss, action_indices

# --- テスト ---
model = ActionDiscoveryModel(in_channels=3, hidden_dim=64, codebook_size=8)
ft, ft1, _ = dataset[0]
x_pred, vq_loss, act_idx = model(ft.unsqueeze(0), ft1.unsqueeze(0))
print(f"ActionDiscoveryModel テスト:")
print(f"  入力フレーム: {ft.shape}")
print(f"  予測フレーム: {x_pred.shape}")
print(f"  発見された行動: {act_idx.item()}")
total_params = sum(p.numel() for p in model.parameters())
print(f"  総パラメータ数: {total_params:,}")

<a id="section5"></a>
## 5. 潜在行動の学習

### 📊 学習の仕組み

LatentActionEncoderの学習は、以下の手順で行います：

1. フレーム対 $(x_t, x_{t+1})$ を入力
2. LatentActionEncoder が離散行動トークン $z_a$ を抽出
3. フレームエンコーダで $x_t$ を潜在表現にエンコード
4. $x_t$ の潜在表現 + $z_a$ から $x_{t+1}$ を予測（デコーダ）
5. 損失 = 再構成損失（$x_{t+1}$ の予測精度）+ VQ損失

**行動ラベルは一切使わない**ことに注目してください。
再構成の過程で「何が変わったか」を表す情報が自然に $z_a$ に集約されます。

In [ ]:
# ============================================================
# サンプルフレーム対の可視化
# ============================================================

fig, axes = plt.subplots(4, 6, figsize=(15, 10))

for row, action_id in enumerate(range(4)):
    # 各行動のサンプルを取得
    indices = np.where(dataset.true_actions == action_id)[0][:3]
    
    for col_offset, idx in enumerate(indices):
        ft, ft1, a = dataset[idx]
        
        # フレームt
        ax = axes[row, col_offset * 2]
        ax.imshow(ft.permute(1, 2, 0).numpy())
        ax.set_title(f'$x_t$', fontsize=10)
        ax.axis('off')
        
        # フレームt+1
        ax = axes[row, col_offset * 2 + 1]
        ax.imshow(ft1.permute(1, 2, 0).numpy())
        ax.set_title(f'$x_{{t+1}}$', fontsize=10)
        ax.axis('off')
    
    # 行動名を左側に表示
    action_name = GridWorldDataset.ACTION_NAMES[action_id]
    axes[row, 0].set_ylabel(f'行動: {action_name} ({action_id})',
                             fontsize=12, fontweight='bold')

plt.suptitle('グリッドワールド: フレーム対と真の行動\n'
             '(赤い四角がエージェント、矢印方向に移動)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# グリッドワールド データ生成
# ============================================================

class GridWorldDataset(Dataset):
    """グリッドワールド: エージェントが4方向に移動する合成動画データ
    
    - フレーム: 16x16 RGB
    - エージェント: 3x3の色付き四角
    - 背景: 暗い格子模様
    - 行動: 上(0), 下(1), 左(2), 右(3)
    """
    
    # 行動の名前
    ACTION_NAMES = ['上', '下', '左', '右']
    
    def __init__(self, num_pairs=2000, grid_size=16, agent_size=3, step_size=2, seed=42):
        super().__init__()
        self.grid_size = grid_size
        self.agent_size = agent_size
        self.step_size = step_size
        
        rng = np.random.RandomState(seed)
        
        # エージェントの色（明るい赤）
        self.agent_color = np.array([0.9, 0.2, 0.2])
        
        # フレーム対と行動ラベルを生成
        self.frame_t = []
        self.frame_t1 = []
        self.true_actions = []
        
        for i in range(num_pairs):
            # ランダムなエージェント位置
            max_pos = grid_size - agent_size
            ax = rng.randint(step_size, max_pos - step_size)
            ay = rng.randint(step_size, max_pos - step_size)
            
            # ランダムな行動を選択
            action = rng.randint(0, 4)
            
            # 行動に応じた移動量
            dx, dy = 0, 0
            if action == 0:    # 上
                dy = -self.step_size
            elif action == 1:  # 下
                dy = self.step_size
            elif action == 2:  # 左
                dx = -self.step_size
            elif action == 3:  # 右
                dx = self.step_size
            
            # フレームを生成
            frame_t = self._render_frame(ax, ay)
            frame_t1 = self._render_frame(ax + dx, ay + dy)
            
            self.frame_t.append(torch.tensor(frame_t, dtype=torch.float32))
            self.frame_t1.append(torch.tensor(frame_t1, dtype=torch.float32))
            self.true_actions.append(action)
        
        self.true_actions = np.array(self.true_actions)
        print(f"グリッドワールドデータ生成完了: {num_pairs}対")
        print(f"  行動分布: {dict(zip(*np.unique(self.true_actions, return_counts=True)))}")
    
    def _render_frame(self, ax, ay):
        """フレームをレンダリング
        
        Args:
            ax, ay: エージェントの位置（左上座標）
        Returns:
            (3, H, W) のフレーム配列
        """
        frame = np.zeros((3, self.grid_size, self.grid_size))
        
        # 背景: 薄い格子模様
        for r in range(self.grid_size):
            for c in range(self.grid_size):
                if (r + c) % 4 < 2:
                    frame[:, r, c] = 0.15
                else:
                    frame[:, r, c] = 0.1
        
        # エージェントを描画
        ay_c = np.clip(ay, 0, self.grid_size - self.agent_size)
        ax_c = np.clip(ax, 0, self.grid_size - self.agent_size)
        for ch in range(3):
            frame[ch, ay_c:ay_c+self.agent_size, ax_c:ax_c+self.agent_size] = self.agent_color[ch]
        
        return frame
    
    def __len__(self):
        return len(self.frame_t)
    
    def __getitem__(self, idx):
        return self.frame_t[idx], self.frame_t1[idx], self.true_actions[idx]

# --- データセット生成 ---
dataset = GridWorldDataset(num_pairs=2000, seed=42)

<a id="section4"></a>
## 4. 合成データの準備 — グリッドワールド

### 📊 シンプルなグリッドワールド動画

Genieの学習メカニズムを検証するために、シンプルな**グリッドワールド**を合成します。

- **16x16ピクセル**のRGBフレーム
- **エージェント**: 色付きの3x3ピクセルの四角形
- **4方向の移動**: 上(0)・下(1)・左(2)・右(3)
- **2000個のフレーム対** $(x_t, x_{t+1})$ を生成
- 各フレーム対には**真の行動ラベル**が付与される（ただし学習時には使わない!）

重要: **行動ラベルは評価にのみ使用**し、モデルの学習には一切使いません。
モデルがラベルなしで行動を正しく発見できるかを検証します。

In [ ]:
# ============================================================
# LatentActionEncoder の実装
# ============================================================

class LatentActionEncoder(nn.Module):
    """潜在行動エンコーダ
    
    2つの連続フレーム (x_t, x_{t+1}) を受け取り、
    フレーム間の変化を離散的な行動トークンとしてエンコードする。
    
    Args:
        in_channels (int): フレームのチャンネル数（RGB=3, グレースケール=1）
        hidden_dim (int): 隠れ層の次元
        codebook_size (int): 発見する行動の種類数
    """
    
    def __init__(self, in_channels=3, hidden_dim=64, codebook_size=8):
        super().__init__()
        self.codebook_size = codebook_size
        
        # CNNエンコーダ: 2フレームをチャンネル方向に結合して入力
        # 入力: (B, in_channels*2, H, W) → 出力: (B, hidden_dim)
        self.encoder = nn.Sequential(
            # (B, C*2, 16, 16) → (B, 32, 8, 8)
            nn.Conv2d(in_channels * 2, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            # (B, 32, 8, 8) → (B, 64, 4, 4)
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            # (B, 64, 4, 4) → (B, 64, 2, 2)
            nn.Conv2d(64, hidden_dim, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            # Global Average Pooling → (B, hidden_dim)
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
        )
        
        # ベクトル量子化: 連続ベクトル → 離散行動トークン
        self.vq = VectorQuantizer(
            num_embeddings=codebook_size,
            embedding_dim=hidden_dim,
            commitment_cost=0.25
        )
    
    def forward(self, x_t, x_t1):
        """順伝播
        
        Args:
            x_t:  (B, C, H, W) 現在のフレーム
            x_t1: (B, C, H, W) 次のフレーム
            
        Returns:
            z_q: (B, hidden_dim) 量子化された行動ベクトル
            vq_loss: VQ損失
            action_indices: (B,) 離散行動インデックス（0 ~ codebook_size-1）
        """
        # 2フレームをチャンネル方向に結合
        x_pair = torch.cat([x_t, x_t1], dim=1)  # (B, C*2, H, W)
        
        # CNN で特徴抽出
        z_e = self.encoder(x_pair)  # (B, hidden_dim)
        
        # ベクトル量子化
        z_q, vq_loss, action_indices = self.vq(z_e)
        
        return z_q, vq_loss, action_indices

# --- テスト ---
lat_enc = LatentActionEncoder(in_channels=3, hidden_dim=64, codebook_size=8)
x_t = torch.randn(4, 3, 16, 16)
x_t1 = torch.randn(4, 3, 16, 16)
z_q, vq_loss, action_idx = lat_enc(x_t, x_t1)

print("LatentActionEncoder テスト:")
print(f"  フレーム入力: {x_t.shape}")
print(f"  行動ベクトル z_q: {z_q.shape}")
print(f"  VQ損失: {vq_loss.item():.4f}")
print(f"  離散行動インデックス: {action_idx.tolist()}")
print(f"  行動の種類数: {lat_enc.codebook_size}")

total_params = sum(p.numel() for p in lat_enc.parameters())
print(f"  パラメータ数: {total_params:,}")

<a id="section3"></a>
## 3. LatentActionEncoder — 潜在行動の抽出

### 📊 核心のアイデア

Genieの最も革新的な部分は**Latent Action Model**です。
このモデルは、**2つの連続フレーム $(x_t, x_{t+1})$ を入力**とし、
フレーム間の変化を**離散的な行動トークン $z_a$** として抽出します。

```
フレームt    フレームt+1
  x_t    +    x_{t+1}
   |            |
   +----+-------+
        |
   [CNN Encoder]
        |
   z_e (連続ベクトル)
        |
   [VQ量子化]
        |
   z_a (離散行動トークン)  ← 行動ラベルなしで発見!
```

ポイント:
- 行動ラベルは一切使わない
- VQ量子化により離散トークンとして行動を表現
- コードブックサイズ = 発見される行動の種類数（ここでは8）

In [ ]:
# ============================================================
# VQ-VAEの結果を可視化
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- (a) 訓練損失 ---
axes[0].plot(losses, 'b-', linewidth=2)
axes[0].set_xlabel('エポック', fontsize=12)
axes[0].set_ylabel('損失', fontsize=12)
axes[0].set_title('VQ-VAE 訓練損失', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# --- (b) コードブック使用頻度 ---
vqvae.eval()
with torch.no_grad():
    _, _, all_indices = vqvae(data.to(device))
    all_indices = all_indices.cpu().numpy()

# ヒストグラム: どのコードブックエントリが使われたか
unique, counts = np.unique(all_indices, return_counts=True)
axes[1].bar(unique, counts, color='#3498DB', edgecolor='black', alpha=0.8)
axes[1].set_xlabel('コードブックインデックス', fontsize=12)
axes[1].set_ylabel('使用回数', fontsize=12)
axes[1].set_title('コードブック使用頻度', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

# --- (c) コードブックベクトルの可視化（PCA→2D） ---
codebook = vqvae.vq.embedding.weight.detach().cpu().numpy()  # (K, D)
# 簡易PCA: 上位2成分
from numpy.linalg import svd
codebook_centered = codebook - codebook.mean(axis=0)
U, S, Vt = svd(codebook_centered, full_matrices=False)
codebook_2d = codebook_centered @ Vt[:2].T

# 使われているコードブックエントリを強調
used_mask = np.isin(np.arange(len(codebook)), unique)
axes[2].scatter(codebook_2d[~used_mask, 0], codebook_2d[~used_mask, 1],
                c='gray', s=60, alpha=0.4, label='未使用', edgecolors='black', linewidth=0.5)
axes[2].scatter(codebook_2d[used_mask, 0], codebook_2d[used_mask, 1],
                c='#E74C3C', s=100, alpha=0.9, label='使用中', edgecolors='black', linewidth=1)
for i in unique:
    axes[2].annotate(str(i), (codebook_2d[i, 0], codebook_2d[i, 1]),
                     fontsize=8, ha='center', va='bottom')
axes[2].set_xlabel('第1主成分', fontsize=12)
axes[2].set_ylabel('第2主成分', fontsize=12)
axes[2].set_title('コードブックベクトル（PCA 2D）', fontsize=13, fontweight='bold')
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"使用されたコードブックエントリ: {len(unique)}/{vqvae.vq.num_embeddings}")
print(f"コードブック利用率: {len(unique)/vqvae.vq.num_embeddings*100:.1f}%")

In [ ]:
# ============================================================
# 簡易VQ-VAEの構築とテスト
# ============================================================

class SimpleVQVAE(nn.Module):
    """簡易VQ-VAE（1次元データ用のテスト）
    
    VQ-VAEの動作確認のためのミニモデル。
    入力データをコードブックの離散ベクトルに圧縮・復元する。
    """
    
    def __init__(self, input_dim=16, hidden_dim=64, num_embeddings=32):
        super().__init__()
        
        # エンコーダ: 入力 → 連続潜在表現
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        
        # ベクトル量子化
        self.vq = VectorQuantizer(
            num_embeddings=num_embeddings,
            embedding_dim=hidden_dim,
            commitment_cost=0.25
        )
        
        # デコーダ: 離散潜在表現 → 再構成
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )
    
    def forward(self, x):
        """順伝播
        
        Returns:
            x_recon: 再構成された入力
            vq_loss: VQ損失
            indices: 使用されたコードブックインデックス
        """
        z_e = self.encoder(x)                       # エンコード
        z_q, vq_loss, indices = self.vq(z_e)        # 量子化
        x_recon = self.decoder(z_q)                  # デコード
        return x_recon, vq_loss, indices

# --- 合成データでテスト ---
# 8つのクラスタを持つ合成データを生成
torch.manual_seed(42)
n_samples = 1000
input_dim = 16
n_clusters = 8

# 各クラスタの中心を生成
centers = torch.randn(n_clusters, input_dim) * 3
# 各サンプルをランダムなクラスタに割り当て
labels = torch.randint(0, n_clusters, (n_samples,))
data = centers[labels] + torch.randn(n_samples, input_dim) * 0.3

print(f"合成データ: {data.shape}")
print(f"クラスタ数: {n_clusters}")

# VQ-VAEを訓練
vqvae = SimpleVQVAE(input_dim=input_dim, hidden_dim=64, num_embeddings=32).to(device)
optimizer = torch.optim.Adam(vqvae.parameters(), lr=1e-3)

# 訓練ループ
losses = []
for epoch in range(50):
    vqvae.train()
    x_batch = data.to(device)
    x_recon, vq_loss, indices = vqvae(x_batch)
    
    recon_loss = F.mse_loss(x_recon, x_batch)
    loss = recon_loss + vq_loss
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1}/50 | Loss: {loss.item():.4f} "
              f"(Recon: {recon_loss.item():.4f}, VQ: {vq_loss.item():.4f})")

print("VQ-VAE 訓練完了")

### VQ-VAEの損失関数

VQ-VAEの全体損失は3つの成分から構成されます：

$$\mathcal{L} = \underbrace{\|x - \hat{x}\|^2}_{\text{再構成損失}} + \underbrace{\|z_q - \text{sg}(z_e)\|^2}_{\text{コードブック損失}} + \underbrace{\beta \|z_e - \text{sg}(z_q)\|^2}_{\text{コミットメント損失}}$$

- $\text{sg}(\cdot)$: Stop Gradient（勾配を止める操作）
- **再構成損失**: デコーダの復元精度
- **コードブック損失**: コードブックベクトルをエンコーダ出力に引き寄せる
- **コミットメント損失**: エンコーダ出力がコードブックから離れすぎないようにする

In [ ]:
# ============================================================
# VectorQuantizer の実装
# ============================================================

class VectorQuantizer(nn.Module):
    """ベクトル量子化層
    
    連続的なエンコーダ出力を、離散的なコードブックベクトルに変換する。
    
    Args:
        num_embeddings (int): コードブックのサイズ K（離散コードの数）
        embedding_dim (int): 各コードブックベクトルの次元
        commitment_cost (float): コミットメント損失の重み beta
    """
    
    def __init__(self, num_embeddings=32, embedding_dim=64, commitment_cost=0.25):
        super().__init__()
        self.num_embeddings = num_embeddings   # K: コードブックサイズ
        self.embedding_dim = embedding_dim     # D: 埋め込み次元
        self.commitment_cost = commitment_cost # beta: コミットメント損失の重み
        
        # コードブック: K個のD次元ベクトル
        # 一様分布 [-1/K, 1/K] で初期化
        self.embedding = nn.Embedding(num_embeddings, embedding_dim)
        self.embedding.weight.data.uniform_(-1.0 / num_embeddings, 1.0 / num_embeddings)
    
    def forward(self, z_e):
        """ベクトル量子化の順伝播
        
        Args:
            z_e: (B, D) エンコーダの出力（連続値）
            
        Returns:
            z_q: (B, D) 量子化されたベクトル
            vq_loss: VQ損失（コードブック損失 + コミットメント損失）
            encoding_indices: (B,) 選択されたコードブックのインデックス
        """
        # --- Step 1: 最近傍のコードブックベクトルを探索 ---
        # z_eとコードブック全体の距離を計算
        # |z_e - e_k|^2 = |z_e|^2 + |e_k|^2 - 2 * z_e . e_k
        distances = (
            torch.sum(z_e ** 2, dim=1, keepdim=True)          # (B, 1)
            + torch.sum(self.embedding.weight ** 2, dim=1)     # (K,)
            - 2 * torch.matmul(z_e, self.embedding.weight.t()) # (B, K)
        )
        
        # 最も近いコードブックベクトルのインデックス
        encoding_indices = torch.argmin(distances, dim=1)  # (B,)
        
        # 量子化ベクトルを取得
        z_q = self.embedding(encoding_indices)  # (B, D)
        
        # --- Step 2: 損失の計算 ---
        # コードブック損失: コードブックをエンコーダ出力に近づける
        codebook_loss = F.mse_loss(z_q, z_e.detach())
        
        # コミットメント損失: エンコーダ出力をコードブックに近づける
        commitment_loss = F.mse_loss(z_e, z_q.detach())
        
        # VQ損失の合計
        vq_loss = codebook_loss + self.commitment_cost * commitment_loss
        
        # --- Step 3: Straight-Through Estimator ---
        # 順伝播: z_q を使う（離散化された値）
        # 逆伝播: 勾配を z_e にそのまま通す
        z_q_st = z_e + (z_q - z_e).detach()
        
        return z_q_st, vq_loss, encoding_indices

# --- テスト ---
vq = VectorQuantizer(num_embeddings=32, embedding_dim=64)
z_e_test = torch.randn(8, 64)  # バッチ8, 64次元
z_q_test, loss_test, indices_test = vq(z_e_test)

print("VectorQuantizer テスト:")
print(f"  入力 z_e:     {z_e_test.shape}")
print(f"  出力 z_q:     {z_q_test.shape}")
print(f"  VQ損失:       {loss_test.item():.4f}")
print(f"  選択インデックス: {indices_test.tolist()}")
print(f"  コードブックサイズ: {vq.num_embeddings}")
print(f"  勾配が通るか: {z_q_test.requires_grad}")

<a id="section2"></a>
## 2. VQ-VAE（Vector Quantized VAE）の実装

### 📊 なぜVQ-VAEが必要か？

通常のVAE（Notebook 36）では、潜在空間は**連続値**です。
しかしGenieでは、行動を**離散トークン**として扱いたい理由があります：

1. **ゲーム操作は本質的に離散**: 上・下・左・右のような離散的な選択
2. **Transformerとの相性**: 言語モデルと同様にトークン列として扱える
3. **解釈性**: 離散コードブックの各エントリに意味を対応づけやすい

### VQ-VAEの仕組み

```
入力 x → [Encoder] → z_e (連続) → [VQ: 最近傍探索] → z_q (離散) → [Decoder] → x_hat
```

**ベクトル量子化（Vector Quantization）**:
- K個の学習可能な埋め込みベクトル $\{e_1, e_2, \ldots, e_K\}$ を持つ「コードブック」
- エンコーダ出力 $z_e$ に最も近いコードブックベクトルを選択: $z_q = e_k$ where $k = \arg\min_j \|z_e - e_j\|$
- **Straight-Through Estimator**: 順伝播では $z_q$ を使い、逆伝播では勾配を $z_e$ にそのまま通す

In [ ]:
# ============================================================
# Genieのアーキテクチャ概要図
# ============================================================

def draw_genie_architecture():
    """Genieの3つのコンポーネントとデータフローを図示"""
    fig, ax = plt.subplots(figsize=(16, 8))
    ax.set_xlim(0, 16)
    ax.set_ylim(0, 8)
    ax.axis('off')

    # --- タイトル ---
    ax.text(8, 7.5, 'Genie アーキテクチャ概要', ha='center', fontsize=16, fontweight='bold')

    # --- Video Tokenizer (左) ---
    rect_vt = plt.Rectangle((0.5, 3.5), 3.5, 2.5, color='#AED6F1', ec='black', lw=2, zorder=2)
    ax.add_patch(rect_vt)
    ax.text(2.25, 5.5, 'Video Tokenizer', ha='center', fontsize=11, fontweight='bold')
    ax.text(2.25, 4.9, '(VQ-VAE)', ha='center', fontsize=10, color='gray')
    ax.text(2.25, 4.3, 'フレーム → 潜在トークン', ha='center', fontsize=9)
    ax.text(2.25, 3.8, 'フレーム復元も担当', ha='center', fontsize=9)

    # フレーム入力（上部）
    for i, label in enumerate(['$x_t$', '$x_{t+1}$']):
        rect_f = plt.Rectangle((1.0 + i * 1.8, 6.5), 1.2, 0.7, color='#FADBD8',
                                ec='black', lw=1.5, zorder=2)
        ax.add_patch(rect_f)
        ax.text(1.6 + i * 1.8, 6.85, label, ha='center', fontsize=11)
    # 矢印: フレーム → Tokenizer
    ax.annotate('', xy=(1.6, 6.0), xytext=(1.6, 6.5),
                arrowprops=dict(arrowstyle='->', lw=2, color='black'))
    ax.annotate('', xy=(3.4, 6.0), xytext=(3.4, 6.5),
                arrowprops=dict(arrowstyle='->', lw=2, color='black'))

    # --- Latent Action Model (中央) ---
    rect_la = plt.Rectangle((5.5, 3.5), 4.5, 2.5, color='#D5F5E3', ec='black', lw=2, zorder=2)
    ax.add_patch(rect_la)
    ax.text(7.75, 5.5, 'Latent Action Model', ha='center', fontsize=11, fontweight='bold')
    ax.text(7.75, 4.9, '(行動ラベル不要!)', ha='center', fontsize=10, color='red')
    ax.text(7.75, 4.3, '$(x_t, x_{t+1})$ → 離散行動 $z_a$', ha='center', fontsize=9)
    ax.text(7.75, 3.8, 'VQ量子化で離散トークン化', ha='center', fontsize=9)

    # 矢印: Tokenizer → Latent Action Model
    ax.annotate('', xy=(5.5, 4.75), xytext=(4.0, 4.75),
                arrowprops=dict(arrowstyle='->', lw=2, color='#2E86C1'))

    # 離散行動出力（下部）
    rect_za = plt.Rectangle((6.75, 1.5), 2.0, 0.8, color='#F9E79F',
                              ec='black', lw=2, zorder=2)
    ax.add_patch(rect_za)
    ax.text(7.75, 1.9, '離散行動 $z_a$', ha='center', fontsize=10, fontweight='bold')
    ax.annotate('', xy=(7.75, 2.3), xytext=(7.75, 3.5),
                arrowprops=dict(arrowstyle='->', lw=2, color='#27AE60'))

    # --- Dynamics Model (右) ---
    rect_dm = plt.Rectangle((11.5, 3.5), 4.0, 2.5, color='#E8DAEF', ec='black', lw=2, zorder=2)
    ax.add_patch(rect_dm)
    ax.text(13.5, 5.5, 'Dynamics Model', ha='center', fontsize=11, fontweight='bold')
    ax.text(13.5, 4.9, '(ST-Transformer)', ha='center', fontsize=10, color='gray')
    ax.text(13.5, 4.3, '潜在状態 + $z_a$ → 次の状態', ha='center', fontsize=9)
    ax.text(13.5, 3.8, '自己回帰的に未来を予測', ha='center', fontsize=9)

    # 矢印: Latent Action → Dynamics
    ax.annotate('', xy=(11.5, 4.75), xytext=(10.0, 4.75),
                arrowprops=dict(arrowstyle='->', lw=2, color='#8E44AD'))
    # 矢印: 離散行動 → Dynamics
    ax.annotate('', xy=(11.5, 3.7), xytext=(8.75, 1.9),
                arrowprops=dict(arrowstyle='->', lw=2, color='#F39C12', ls='--'))

    # 予測フレーム出力
    rect_pred = plt.Rectangle((12.5, 1.5), 2.0, 0.8, color='#FADBD8',
                                ec='black', lw=2, zorder=2)
    ax.add_patch(rect_pred)
    ax.text(13.5, 1.9, '予測 $\\hat{x}_{t+1}$', ha='center', fontsize=10, fontweight='bold')
    ax.annotate('', xy=(13.5, 2.3), xytext=(13.5, 3.5),
                arrowprops=dict(arrowstyle='->', lw=2, color='#8E44AD'))

    # --- 推論時のフロー注釈 ---
    ax.text(8, 0.7, '【推論時】 ユーザーが行動トークン $z_a$ を選択 → Dynamics Model が次フレームを生成 → 繰り返し',
            ha='center', fontsize=10, style='italic',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', ec='orange', lw=1.5))

    plt.tight_layout()
    plt.show()

draw_genie_architecture()

<a id="section1"></a>
## 1. Genieの革新 — ラベルなしで行動を発見する

### 🤔 従来の強化学習とGenieの違い

従来の強化学習やゲームAIでは、**環境から明示的な行動ラベル**が与えられます：
- 上・下・左・右・ジャンプ...

しかし、インターネット上の大量の動画には**行動ラベルが付いていません**。
人がゲームをプレイしている動画を見ても、どのボタンが押されたかは分かりません。

### Genieの核心的アイデア

> **「2つの連続フレーム間の変化こそが行動である」**

Genie（Generative Interactive Environment）は、2024年にGoogle DeepMindが発表した
画期的なモデルです。**ラベルなし動画だけ**から以下を同時に学習します：

| コンポーネント | 役割 |
|---------------|------|
| **Latent Action Model** | 連続2フレームから「何が起きたか」を離散トークンとして抽出 |
| **Video Tokenizer** | フレームをコンパクトな潜在表現に圧縮（VQ-VAE） |
| **Dynamics Model** | 現在のフレーム + 行動トークン → 次のフレームを予測 |

### 応用：1枚の画像からプレイ可能な世界を生成

学習後は、**1枚の画像を入力**するだけで、ユーザーが操作可能なインタラクティブな
仮想世界を生成できます。これは従来のゲームエンジンなしで実現される点が革新的です。

In [ ]:
# ============================================================
# 環境設定
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.manifold import TSNE
import warnings
import math

warnings.filterwarnings('ignore')

# --- 日本語フォント設定 ---
def setup_japanese_font():
    japanese_fonts = ['Hiragino Sans', 'Yu Gothic', 'Noto Sans CJK JP', 'IPAexGothic']
    available = set(f.name for f in fm.fontManager.ttflist)
    for f in japanese_fonts:
        if f in available:
            plt.rcParams['font.family'] = f
            plt.rcParams['axes.unicode_minus'] = False
            return f
    return None

font_used = setup_japanese_font()
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

# --- 再現性 ---
torch.manual_seed(42)
np.random.seed(42)

# --- デバイス ---
device = torch.device('cuda' if torch.cuda.is_available() else
                       'mps' if torch.backends.mps.is_available() else 'cpu')

print(f'PyTorch version: {torch.__version__}')
print(f'デバイス: {device}')
print('ライブラリのインポート完了')

## 目次

1. [Genieの革新 — ラベルなしで行動を発見する](#section1)
2. [VQ-VAE（Vector Quantized VAE）の実装](#section2)
3. [LatentActionEncoder — 潜在行動の抽出](#section3)
4. [合成データの準備 — グリッドワールド](#section4)
5. [潜在行動の学習](#section5)
6. [DynamicsModel — 次フレーム予測](#section6)
7. [SimpleGenie — 統合モデル](#section7)
8. [潜在行動空間の分析](#section8)
9. [まとめと自己評価クイズ](#summary)

# 第144章: Genie — 動画から操作可能な世界を学ぶ

## 📋 この章で学ぶこと

この章を終えると、以下ができるようになります：

- [ ] 行動ラベルなしで潜在行動空間を学習できる
- [ ] VQ-VAE（ベクトル量子化VAE）を実装できる
- [ ] 連続フレーム対から離散行動を発見するエンコーダを構築できる
- [ ] 時空間Transformerによる次フレーム予測モデルを実装できる
- [ ] 潜在行動空間をt-SNEで分析・可視化できる

## 🎯 前提知識

- ✅ Notebook 143（World Modelsの基礎）
- ✅ Notebook 36（VAE — 変分オートエンコーダ）
- ✅ Notebook 132（DiT — Diffusion Transformer）

⏱️ **推定学習時間**: 150-180分
📊 **難易度**: ★★★★★（最上級）
🎓 **カテゴリ**: World Models / インタラクティブ生成